In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import requests
from urllib.parse import quote

In [2]:
HIST_START = pd.Timestamp("2025-10-13")
WEEK_END = pd.Timestamp("2026-10-18")
DAYS = pd.date_range(HIST_START, WEEK_END, freq="D")
UA = {"User-Agent": "DS3-IndicatorLab/1.0 (jms.mulyono@gmail.com)"}

In [10]:
TITLES = {
    "AAPL": "Apple_Inc.",
    "MSFT": "Microsoft",
    "NVDA": "Nvidia",
    "TSLA": "Tesla,_Inc.",
    "AMZN": "Amazon_(company)",
    "META": "Meta_Platforms",
    "GOOGL": "Alphabet_Inc.",
    "AMD": "AMD",
    "NFLX": "Netflix",
    "JPM": "JPMorgan_Chase",
}



In [11]:
r = requests.get(
    "https://en.wikipedia.org/w/api.php",
    headers=UA,
    params={
        "action": "query",
        "titles": "|".join(TITLES.values()),
        "redirects": 1,
        "format": "json",
        "formatversion": 2,
    },
)

In [12]:

q = r.json()["query"]
print("redirects:", q.get("redirects"))
print("missing:", [p["title"] for p in q["pages"] if p.get("missing")])
print([p["title"] for p in q["pages"]])

redirects: None
missing: []
['Apple Inc.', 'AMD', 'Microsoft', 'Nvidia', 'Amazon (company)', 'Netflix', 'JPMorgan Chase', 'Tesla, Inc.', 'Alphabet Inc.', 'Meta Platforms']


In [17]:
%load_ext autoreload
%autoreload 2
from src.wiki_attention import pageviews

v = pageviews("Apple_Inc.", HIST_START, WEEK_END)
print(len(v), v["2026-09-09"])  

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
371 22115


In [18]:
from pathlib import Path
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def load_views(ticker, title):
    f = RAW_DIR / f"{ticker}_views.csv"
    if f.exists():
        return pd.read_csv(f, index_col=0, parse_dates=True).iloc[:, 0]
    s = pageviews(title, HIST_START, WEEK_END).rename(ticker)
    s.to_csv(f)
    return s

daily = pd.concat([load_views(t, ti) for t, ti in TITLES.items()], axis=1)
print(daily.shape, daily.loc["2026-10-16", "TSLA"])   

(371, 10) 12751


In [19]:
weekly = (
    daily.resample("W-SUN").sum()
    .rename_axis("week_end")
    .stack()
    .rename("views")
    .rename_axis(["week_end", "ticker"])
    .reset_index()
)
print(weekly.shape)   # expect (530, 3): 53 weeks x 10 tickers
weekly.head()

(530, 3)


,week_end,ticker,views
0,2025-10-19,AAPL,50301
1,2025-10-19,MSFT,46449
2,2025-10-19,NVDA,54068
3,2025-10-19,TSLA,27833
4,2025-10-19,AMZN,48947


In [20]:
from src.wiki_attention import flag_spikes

wk = daily.resample("W-SUN").sum()
spikes = pd.concat(
    [flag_spikes(wk[t]).assign(ticker=t) for t in wk.columns]
).rename_axis("week_end").reset_index()
print(spikes["spike"].sum(), "spike weeks")
spikes[spikes["spike"]].head(10)

17 spike weeks


,week_end,views,baseline,abnormal,rz,spike,ticker
27,2026-04-26,103777,44026.5,0.857453,9.904378,True,AAPL
46,2026-09-06,90882,39677.0,0.828790,9.574492,True,AAPL
47,2026-09-13,95591,39677.0,0.879307,10.155908,True,AAPL
62,2025-12-21,128884,38301.0,1.213437,9.765445,True,MSFT
136,2026-05-17,99537,38939.0,0.938533,4.192795,True,NVDA
137,2026-05-24,67816,38939.0,0.554802,2.602891,True,NVDA
139,2026-06-07,64841,39926.5,0.484898,2.313260,True,NVDA
151,2026-08-30,48519,31976.0,0.416970,2.031815,True,NVDA
168,2025-12-21,64010,26756.5,0.872262,5.009909,True,TSLA
169,2025-12-28,66992,27705.0,0.882960,5.070328,True,TSLA


In [21]:
import yfinance as yf

px = yf.download("AAPL", start="2026-07-01", end="2026-10-20", auto_adjust=True, progress=False)
close = px["Close"].squeeze()
ret = close.resample("W-SUN").last().pct_change().rename("aapl_ret")

aapl = (
    spikes[spikes["ticker"] == "AAPL"]
    .set_index("week_end")[["views", "baseline", "abnormal", "spike"]]
    .join(ret, how="left")
)
aapl.loc["2026-08-01":"2026-10-18"]

,views,baseline,abnormal,spike,aapl_ret
week_end,,,,,
2026-08-02,39969,40208.0,-0.005962,False,-0.072398
2026-08-09,37395,39553.5,-0.056117,False,0.014308
2026-08-16,37412,39110.5,-0.044399,False,-0.022775
2026-08-23,39385,38374.5,0.025992,False,0.011179
2026-08-30,44032,39261.5,0.114672,False,0.033457
2026-09-06,90882,39677.0,0.828790,True,0.000845
2026-09-13,95591,39677.0,0.879307,True,0.038441
2026-09-20,59037,39677.0,0.397393,False,0.011617
2026-09-27,48057,42000.5,0.134706,False,0.014697


In [22]:
daily["AAPL"].loc["2026-08-28":"2026-09-12"]


2026-08-28     6149
2026-08-29     5739
2026-08-30     6332
2026-08-31    12578
2026-09-01    24268
2026-09-02    18734
2026-09-03    12610
2026-09-04     8994
2026-09-05     7258
2026-09-06     6440
2026-09-07     7550
2026-09-08     8693
2026-09-09    22115
2026-09-10    24292
2026-09-11    13811
2026-09-12    10315
Name: AAPL, dtype: int64

## Findings
- 17 spike weeks across 10 tickers (rule: >=1.5x 8-week median baseline and robust z >= 2.0).
- AAPL spikes in the weeks ending 2026-09-06 (CEO handover, Sep 1-2) and 2026-09-13 (iPhone event, Sep 9-10), each ~2.3-2.4x baseline.
- Returns in those weeks: +0.1% and +3.8%. Only two weeks, so no conclusion about price.
- The Aug 26 event announcement did not move pageviews (week ending 08-30: 1.1x).
- Open: compare against Joshua's Reddit spikes once his table is merged.